%md
### 🔗 Links and Resources
| Method | Description |
|--------|-------------|
| [DataFrame.join()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html?highlight=join#pyspark.sql.DataFrame.join) | Joins two DataFrames using a condition and join type (inner, left, right, outer, anti, etc.) |
| [DataFrame.crossJoin()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.crossJoin.html?highlight=join#pyspark.sql.DataFrame.crossJoin) | Returns the Cartesian product of two DataFrames (every row paired with every other row) |

In [1]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

📌 Creating DataFrames

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define schema for df_1 (Sales) with transaction_id included
schema_sales = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("sales_amount", IntegerType(), True)
])

# Data for df_1: sales records
# Some store_ids (e.g., 106 and 107) are not present in df_2 to demonstrate join gaps.
data_sales = [
    (1001, 103, "2025-01-15", 5000),
    (1002, 104, "2025-01-16", 7000),
    (1003, 105, "2025-01-17", 6500),
    (1004, 106, "2025-01-18", 4800),
    (1005, 107, "2025-01-19", 5300)
]

# Create df_1 with the sales data
df_1 = spark.createDataFrame(data_sales, schema=schema_sales)
df_1.show()

# Define schema for df_2 (Stores)
schema_stores = StructType([
    StructField("id", IntegerType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True)
])

# Data for df_2: stores
data_stores = [
    (101, "Store A", "New York"),
    (102, "Store B", "Los Angeles"),
    (103, "Store C", "Chicago"),
    (104, "Store D", "Houston"),
    (105, "Store E", "Phoenix")
]

# Create df_2 with the stores data
df_2 = spark.createDataFrame(data_stores, schema=schema_stores)
df_2.show()

+--------------+--------+----------+------------+
|transaction_id|store_id| sale_date|sales_amount|
+--------------+--------+----------+------------+
|          1001|     103|2025-01-15|        5000|
|          1002|     104|2025-01-16|        7000|
|          1003|     105|2025-01-17|        6500|
|          1004|     106|2025-01-18|        4800|
|          1005|     107|2025-01-19|        5300|
+--------------+--------+----------+------------+

+---+----------+-----------+
| id|store_name|       city|
+---+----------+-----------+
|101|   Store A|   New York|
|102|   Store B|Los Angeles|
|103|   Store C|    Chicago|
|104|   Store D|    Houston|
|105|   Store E|    Phoenix|
+---+----------+-----------+



📌 Left Join

In [10]:
new_left_join = df_1.join(df_2, df_1.store_id == df_2.id , "left").show()



+--------------+--------+----------+------------+----+----------+-------+
|transaction_id|store_id| sale_date|sales_amount|  id|store_name|   city|
+--------------+--------+----------+------------+----+----------+-------+
|          1001|     103|2025-01-15|        5000| 103|   Store C|Chicago|
|          1002|     104|2025-01-16|        7000| 104|   Store D|Houston|
|          1003|     105|2025-01-17|        6500| 105|   Store E|Phoenix|
|          1004|     106|2025-01-18|        4800|NULL|      NULL|   NULL|
|          1005|     107|2025-01-19|        5300|NULL|      NULL|   NULL|
+--------------+--------+----------+------------+----+----------+-------+



📌 Right Join

In [11]:
right_join_df = df_1.join(df_2, df_1.store_id == df_2.id, "right").show()


+--------------+--------+----------+------------+---+----------+-----------+
|transaction_id|store_id| sale_date|sales_amount| id|store_name|       city|
+--------------+--------+----------+------------+---+----------+-----------+
|          NULL|    NULL|      NULL|        NULL|101|   Store A|   New York|
|          NULL|    NULL|      NULL|        NULL|102|   Store B|Los Angeles|
|          1001|     103|2025-01-15|        5000|103|   Store C|    Chicago|
|          1002|     104|2025-01-16|        7000|104|   Store D|    Houston|
|          1003|     105|2025-01-17|        6500|105|   Store E|    Phoenix|
+--------------+--------+----------+------------+---+----------+-----------+



📌 Inner Join

In [5]:
inner_join_df = df_1.join(df_2, df_1.store_id == df_2.id, "inner")

inner_join_df.show()

+--------------+--------+----------+------------+---+----------+-------+
|transaction_id|store_id| sale_date|sales_amount| id|store_name|   city|
+--------------+--------+----------+------------+---+----------+-------+
|          1001|     103|2025-01-15|        5000|103|   Store C|Chicago|
|          1002|     104|2025-01-16|        7000|104|   Store D|Houston|
|          1003|     105|2025-01-17|        6500|105|   Store E|Phoenix|
+--------------+--------+----------+------------+---+----------+-------+



📌 Full Outer Join

In [6]:
full_join_df = df_1.join(df_2, df_1.store_id == df_2.id, "fullouter")

full_join_df.show()

+--------------+--------+----------+------------+----+----------+-----------+
|transaction_id|store_id| sale_date|sales_amount|  id|store_name|       city|
+--------------+--------+----------+------------+----+----------+-----------+
|          NULL|    NULL|      NULL|        NULL| 101|   Store A|   New York|
|          NULL|    NULL|      NULL|        NULL| 102|   Store B|Los Angeles|
|          1001|     103|2025-01-15|        5000| 103|   Store C|    Chicago|
|          1002|     104|2025-01-16|        7000| 104|   Store D|    Houston|
|          1003|     105|2025-01-17|        6500| 105|   Store E|    Phoenix|
|          1004|     106|2025-01-18|        4800|NULL|      NULL|       NULL|
|          1005|     107|2025-01-19|        5300|NULL|      NULL|       NULL|
+--------------+--------+----------+------------+----+----------+-----------+



📌 Left Anti Join

In [7]:
left_anti_df = df_1.join(df_2, df_1.store_id == df_2.id, "left_anti")

left_anti_df.show()

+--------------+--------+----------+------------+
|transaction_id|store_id| sale_date|sales_amount|
+--------------+--------+----------+------------+
|          1004|     106|2025-01-18|        4800|
|          1005|     107|2025-01-19|        5300|
+--------------+--------+----------+------------+



In [8]:
df_1.crossJoin(df_2).show()

+--------------+--------+----------+------------+---+----------+-----------+
|transaction_id|store_id| sale_date|sales_amount| id|store_name|       city|
+--------------+--------+----------+------------+---+----------+-----------+
|          1001|     103|2025-01-15|        5000|101|   Store A|   New York|
|          1002|     104|2025-01-16|        7000|101|   Store A|   New York|
|          1003|     105|2025-01-17|        6500|101|   Store A|   New York|
|          1004|     106|2025-01-18|        4800|101|   Store A|   New York|
|          1005|     107|2025-01-19|        5300|101|   Store A|   New York|
|          1001|     103|2025-01-15|        5000|102|   Store B|Los Angeles|
|          1002|     104|2025-01-16|        7000|102|   Store B|Los Angeles|
|          1003|     105|2025-01-17|        6500|102|   Store B|Los Angeles|
|          1004|     106|2025-01-18|        4800|102|   Store B|Los Angeles|
|          1005|     107|2025-01-19|        5300|102|   Store B|Los Angeles|